# Validating the implementation of logcial Cliffords

## What have I done?

* Made a simple tool which validates logical Clifford semantics against physical implementation
* Works for $k>1$ codes and also interblock operations
* The theory should work for any stabilizer code (non-drastic) changes needed to accomodate non-CSS.


## Basic example for Steane

In [1]:
from typing import no_type_check


from guppylang import guppy
from guppylang.std.builtins import array
from guppylang.std.mem import mem_swap
from guppylang.std.quantum import qubit, h, cx, rx
from guppylang.std.qsystem.helios import zz_phase, zz_max
from guppylang.std.angles import pi

from guppyft.verifier import compute_verification_signterms, compute_verification_signterms_double_block, compute_stabilizers_single_block, get_expanded_stabilizer_set
from guppyft.verifier.code import StabilizerCode, STEANE

from zixy.qubit import pauli


@guppy
@no_type_check
def steane_logical_h(qs: array[qubit, 1]) -> None:
    h(qs[0])


@guppy
@no_type_check
def steane_physical_h(block: array[qubit, 7]) -> None:
    for i in range(len(block)):
        h(block[i])

In [2]:
@guppy
@no_type_check
def steane_physical_cx(
    first_block: array[qubit, 7], second_block: array[qubit, 7]
) -> None:
    for i in range(len(first_block)):
        cx(first_block[i], second_block[i])


@guppy
@no_type_check
def steane_non_ft_zero() -> array[qubit, 7]:
    """Non fault-tolerant zero state preparation."""
    block = array(qubit() for _ in range(7))

    plus_ids = array(0, 4, 6)
    for i in plus_ids:
        h(block[i])

    cx_pairs = array((0, 1), (4, 5), (6, 3), (6, 5), (4, 2), (0, 3), (4, 1), (3, 2))
    for c, t in cx_pairs:
        cx(block[c], block[t])

    return block

@guppy
@no_type_check
def steane_choi_state(
    unitary: Function[[array[qubit, 7]], None],
) -> tuple[array[qubit, 7], array[qubit, 7]]:
    control_block = steane_non_ft_zero()
    target_block = steane_non_ft_zero()

    steane_physical_h(control_block)
    steane_physical_cx(control_block, target_block)

    unitary(target_block)

    return control_block, target_block

In [3]:
sem, impl = compute_verification_signterms(
    steane_logical_h,
    steane_physical_h,
    code_choi_state_function=steane_choi_state,
    code_definition=STEANE,
)

In [4]:
sem.to_dataframe()

,Component,Coefficient
0,X0 X3 X6 Z11 Z12 Z13,+1
1,X1 X3 X5 Z11 Z12 Z13,+1
2,X2 X3 X5 X6,+1
3,X4 X5 X6 Z11 Z12 Z13,+1
4,Z4 Z5 Z6 X7 X10 X13,+1
5,Z4 Z5 Z6 X8 X10 X12,+1
6,X9 X10 X12 X13,+1
7,Z4 Z5 Z6 X11 X12 X13,+1
8,Z0 Z3 Z4 Z5,+1
9,Z1 Z3 Z4 Z6,+1


In [5]:
impl.to_dataframe()

,Component,Coefficient
0,X0 X3 X6 Z11 Z12 Z13,+1
1,X1 X3 X5 Z11 Z12 Z13,+1
2,X2 X3 X5 X6,+1
3,X4 X5 X6 Z11 Z12 Z13,+1
4,Z4 Z5 Z6 X7 X10 X13,+1
5,Z4 Z5 Z6 X8 X10 X12,+1
6,X9 X10 X12 X13,+1
7,Z4 Z5 Z6 X11 X12 X13,+1
8,Z0 Z3 Z4 Z5,+1
9,Z1 Z3 Z4 Z6,+1


In [6]:
sem == impl

True

## Code definition

```python
@dataclass(frozen=True)
class StabilizerCode:
    num_physical_qubits: int
    num_logical_qubits: int
    distance: int
    generators: pauli.StringSet
    x_logicals: pauli.Strings
    z_logicals: pauli.Strings
```

In [7]:
ICEBERG_4_2_2_GENERATORS = pauli.StringSet.from_cmpnts(
    pauli.Strings.from_str(
        "X0 X1 X2 X3, Z0 Z1 Z2 Z3",
        4,
    )
)

# IMPORTANT: Iceberg codeblock ordering

# [top q1, q2, bottom]

# ICEBERG_4_2_2_X[0]: XI -> XXII
# ICEBERG_4_2_2_X[1]: IX -> XIXI

ICEBERG_4_2_2_X = pauli.Strings.from_str(
    "X0 X1 I2 I3, X0 I1 X2 I3",
    4,
)

# ICEBERG_4_2_2_Z[0]: ZI -> IZIZ
# ICEBERG_4_2_2_Z[1]: IZ -> IIZZ

ICEBERG_4_2_2_Z = pauli.Strings.from_str(
    "I0 Z1 I2 Z3, I0 I1 Z2 Z3",
    4,
)

In [8]:
ICEBERG_4_2_2 = StabilizerCode(
    num_physical_qubits=4,
    num_logical_qubits=2,
    distance=2,
    generators=ICEBERG_4_2_2_GENERATORS,
    x_logicals=ICEBERG_4_2_2_X,
    z_logicals=ICEBERG_4_2_2_Z,
)

## Less trivial iceberg examples

In [9]:
@guppy
@no_type_check
def iceberg_double_h_logical(block: array[qubit, 2]) -> None:
    h(block[0])
    h(block[1])


@guppy
@no_type_check
def iceberg_double_h_physical(block: array[qubit, 4]) -> None:
    h(block[0])
    h(block[1])
    h(block[2])
    h(block[3])
    mem_swap(block[1], block[2])

In [10]:
@guppy
@no_type_check
def iceberg_transversal_cx_logical(
    first_block: array[qubit, 2], second_block: array[qubit, 2]
) -> None:
    for i in range(2):
        cx(first_block[i], second_block[i])


@guppy
@no_type_check
def iceberg_transversal_cx_physical(
    first_block: array[qubit, 4], second_block: array[qubit, 4]
) -> None:
    for i in range(4):
        cx(first_block[i], second_block[i])

In [11]:
@guppy
@no_type_check
def iceberg_non_ft_zero() -> array[qubit, 4]:
    block = array(qubit() for _ in range(4))
    h(block[2])
    cx(block[2], block[1])
    cx(block[2], block[3])
    cx(block[1], block[0])
    return block


@guppy
@no_type_check
def iceberg_choi_state(
    unitary: Function[[array[qubit, 4]], None],
) -> tuple[array[qubit, 4], array[qubit, 4]]:
    control_block = iceberg_non_ft_zero()
    target_block = iceberg_non_ft_zero()

    iceberg_double_h_physical(control_block)
    iceberg_transversal_cx_physical(control_block, target_block)

    unitary(target_block)

    return control_block, target_block



@guppy
@no_type_check
def iceberg_choi_state_double_block(
    unitary: Function[[array[qubit, 4], array[qubit, 4]], None],
) -> tuple[array[qubit, 4], array[qubit, 4], array[qubit, 4], array[qubit, 4]]:
    first_control_block = iceberg_non_ft_zero()
    first_target_block = iceberg_non_ft_zero()
    second_control_block = iceberg_non_ft_zero()
    second_target_block = iceberg_non_ft_zero()

    iceberg_double_h_physical(first_control_block)
    iceberg_double_h_physical(second_control_block)

    iceberg_transversal_cx_physical(first_control_block, first_target_block)
    iceberg_transversal_cx_physical(second_control_block, second_target_block)

    unitary(first_target_block, second_target_block)

    return (
        first_control_block,
        first_target_block,
        second_control_block,
        second_target_block,
    )

In [12]:
@guppy
@no_type_check
def iceberg_intra_block_cx_logical(block: array[qubit, 2]) -> None:
    cx(block[0], block[1])


@guppy
@no_type_check
def iceberg_intra_block_cx_physical(block: array[qubit, 4]) -> None:
    mem_swap(block[3], block[1])

In [13]:
sem, impl = compute_verification_signterms(
        iceberg_intra_block_cx_logical,
        iceberg_intra_block_cx_physical,
        iceberg_choi_state,
        ICEBERG_4_2_2,
    )

sem == impl

True

In [14]:
@guppy
@no_type_check
def iceberg_addressable_rx_minus_half_pi_logical(block: array[qubit, 2]) -> None:
    rx(block[1], -pi / 2)


@guppy
@no_type_check
def iceberg_addressable_rx_minus_half_pi_physical(block: array[qubit, 4]) -> None:
    h(block[0])
    h(block[2])
    zz_phase(block[0], block[2], -pi / 2)
    h(block[0])
    h(block[2])

In [15]:
sem, impl = compute_verification_signterms(
        iceberg_addressable_rx_minus_half_pi_logical,
        iceberg_addressable_rx_minus_half_pi_physical,
        iceberg_choi_state,
        ICEBERG_4_2_2,
    )

sem == impl

True

Now what have I swept under the carpet? Choi states

In [16]:
N = guppy.nat_var("N")

@guppy
@no_type_check
def default_choi_state_preparation(
    unitary_func: Function[[array[qubit, N]], None],
) -> tuple[array[qubit, N], array[qubit, N]]:
    """Prepare a Choi state (unencoded) for a particular n-qubit unitary."""
    control_block = array(qubit() for _ in range(N))
    target_block = array(qubit() for _ in range(N))
    for i in range(N):
        h(control_block[i])
        cx(control_block[i], target_block[i])

    unitary_func(target_block)
    return control_block, target_block


## Trying to explain it step by step

Let's go back to the logical Hadamard in Steane

In [17]:
 # Get the 2k stabilizers for the 2k qubit Choi state encoding the logical operation.
semantic_choi_stabilizers = compute_stabilizers_single_block(
    steane_logical_h,
    choi_state_preparation=default_choi_state_preparation,
    n_func_qubits=STEANE.num_logical_qubits,
)

In [18]:
semantic_choi_stabilizers.to_dataframe()

,Component,Coefficient
0,X0 Z1,+1
1,Z0 X1,+1


In [19]:
# Expand the 2k logical stabilizers to 2k stabilizers of size 2n.
# We also add the 2(n-k) stabilizer generators of our code.
# For each code block there are (n-k) so 2 blocks give us 2(n-k).
# We have 2k + 2(n-k) = 2n stabilizers in total.
expanded_semantic_stabilizers = get_expanded_stabilizer_set(
    semantic_choi_stabilizers, STEANE, num_blocks=1
)

$$
X_L \mapsto XXXXXXX\, \qquad Z_L \mapsto ZZZZZZZ
$$

In [20]:
expanded_semantic_stabilizers.to_dataframe()

,Component,Coefficient
0,X0 X1 X2 X3 X4 X5 X6 Z7 Z8 Z9 Z10 Z11 Z12 Z13,+1
1,Z0 Z1 Z2 Z3 Z4 Z5 Z6 X7 X8 X9 X10 X11 X12 X13,+1
2,X0 X1 X2 X3,+1
3,X1 X2 X4 X5,+1
4,X2 X3 X5 X6,+1
5,Z0 Z1 Z2 Z3,+1
6,Z1 Z2 Z4 Z5,+1
7,Z2 Z3 Z5 Z6,+1
8,X7 X8 X9 X10,+1
9,X8 X9 X11 X12,+1


In [21]:
expanded_semantic_stabilizers.canonicalize_all() # Normalize Clifford tableau
expanded_semantic_stabilizers.to_dataframe()

,Component,Coefficient
0,X0 X3 X6 Z11 Z12 Z13,+1
1,X1 X3 X5 Z11 Z12 Z13,+1
2,X2 X3 X5 X6,+1
3,X4 X5 X6 Z11 Z12 Z13,+1
4,Z4 Z5 Z6 X7 X10 X13,+1
5,Z4 Z5 Z6 X8 X10 X12,+1
6,X9 X10 X12 X13,+1
7,Z4 Z5 Z6 X11 X12 X13,+1
8,Z0 Z3 Z4 Z5,+1
9,Z1 Z3 Z4 Z6,+1


In [22]:
# Calculate the 2n stabilizers of the Choi state encoding the physical operation.
implementation_stabilizers = compute_stabilizers_single_block(
    steane_physical_h,
    steane_choi_state,
    STEANE.num_physical_qubits,
)

In [23]:
implementation_stabilizers.to_dataframe() 

,Component,Coefficient
0,X0 X3 X6 Z11 Z12 Z13,+1
1,Z0 Z3 Z6 X11 X12 X13,+1
2,X1 X3 X5 Z11 Z12 Z13,+1
3,Z1 Z3 Z5 X11 X12 X13,+1
4,X2 X3 X5 X6,+1
5,Z2 Z3 Z5 Z6,+1
6,X4 X5 X6 Z11 Z12 Z13,+1
7,Z4 Z5 Z6 X11 X12 X13,+1
8,X7 X10 X11 X12,+1
9,Z7 Z10 Z11 Z12,+1


In [24]:
implementation_stabilizers.canonicalize_all() # Normalize Clifford tableau
implementation_stabilizers.to_dataframe() 

,Component,Coefficient
0,X0 X3 X6 Z11 Z12 Z13,+1
1,X1 X3 X5 Z11 Z12 Z13,+1
2,X2 X3 X5 X6,+1
3,X4 X5 X6 Z11 Z12 Z13,+1
4,Z4 Z5 Z6 X7 X10 X13,+1
5,Z4 Z5 Z6 X8 X10 X12,+1
6,X9 X10 X12 X13,+1
7,Z4 Z5 Z6 X11 X12 X13,+1
8,Z0 Z3 Z4 Z5,+1
9,Z1 Z3 Z4 Z6,+1


In [25]:
expanded_semantic_stabilizers == implementation_stabilizers

True

## Things to add?

* Testing for non-CSS codes. $[[5, 1, 3]], [[4, 2, 2]] (\text{non-CSS variant})$
* Automatically prepare logical Bell states under the hood... Much friendlier and easier to test non-CSS codes
* Allow validating implementations which use ancilla qubits.
* How feasible is it to do non-Cliffords as well?